In [ ]:
git clone https://github.com/lucasxlu/ComboLoss.git

In [57]:
pip install opencv-python tensorflow imageio scikit-learn torchvision scikit-image pytorchcv

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 532 kB 2.1 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys
import time
import os 
import pandas as pd

import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
from skimage import io
from utils import sort_images
from utils import im_transforms_ratings as im_transforms

sys.path.append('../')
from ComboLoss.models.nets import ComboNet


In [2]:
class FacialPredictor:
    """
    Facial Predictor
    """

    def __init__(self, pretrained_model_path):
        model = ComboNet(num_out=5, backbone_net_name='SEResNeXt50')
        model = model.float()
        device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        model = model.to(device)

        if torch.cuda.device_count() > 1:
            print("We are running on", torch.cuda.device_count(), "GPUs!")
            model = nn.DataParallel(model)
            model.load_state_dict(torch.load(pretrained_model_path))
        else:
            state_dict = torch.load(pretrained_model_path, map_location="cpu")
            from collections import OrderedDict
            new_state_dict = OrderedDict()
            for k, v in state_dict.items():
                name = k[7:]  # remove `module.`
                new_state_dict[name] = v
            model.load_state_dict(new_state_dict)

        model.to(device)
        model.eval()

        self.device = device
        self.model = model

    def infer(self, img_file):
        img = io.imread(img_file)
        img = Image.fromarray(img.astype(np.uint8))

        img = im_transforms(img)
        img.unsqueeze_(0)
        img = img.to(self.device)

        score, cls = self.model(img)

        return float(score.to('cpu').detach().item())


In [3]:
fbp = FacialPredictor(pretrained_model_path='checkpoints/ComboNet_SCUTFBP5500.pth')

In [4]:
indices = sort_images(os.listdir("../images/ratings/"))

In [5]:
ratings = []
for img_idx in range(len(indices)):
    image_path = f"../images/ratings/im{indices[img_idx]}.jpg"
    rating = fbp.infer(image_path)
    ratings.append(rating)

In [6]:
images = [f'im{x}.jpg' for x in indices]

In [7]:
# Create dataframe
df = pd.DataFrame({
    "image_name": images,
    "image_index": indices,
    "score": np.array(ratings)
})

# Save to CSV
df.to_csv("results/ratings.csv", index=False)